### Import modules

In [57]:
from pathlib import Path
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.problems import get_problem
from pymoo.optimize import minimize

import pandas as pd
import numpy as np

### Preliminary: Nodes name and Profit score

In [ ]:
DIR = Path("../data")

# dataframe of .csv
poi = pd.read_csv(DIR / "POI-Edin.csv", sep=";").query("poiID != 26")
cost = pd.read_csv(DIR / "costProfCat-EdinPOI-all.csv", sep=";").query("`from` != 26 and to != 26")
visits = pd.read_csv(DIR / "userVisits-Edin.csv", sep=";")

time_budget = 420

ids = []
for p in poi.poiID:
    if p != 17:
        ids.append(p)
nodes = ids + [17]

nodes_names = poi.set_index("poiID").loc[ids, "poiName"].to_numpy()
profit_score = cost.groupby("to")["profit"].first().loc[ids].to_numpy(float)

#### Mini section: Walking time cost

In [ ]:
WALKING_SPEED = 4000 # 4 km/hr -> 4000 m/hr

TRAVEL_TIME_MATRIX = (cost.pivot(index="from", columns="to", values="cost")
                      .reindex(index=nodes, columns=nodes)
                      .fillna(0)
                      .to_numpy() / (WALKING_SPEED / 60))

TRAVEL_TIME_MATRIX

array([[  0.        ,  27.27374209,  10.50611665,  24.68257209,
        195.71648072, 209.04081511,  37.11452051,   9.50161306,
          9.45436436,  20.65937957,  20.50734404,   4.42349021,
         14.06925805,   9.24032932,   6.68850083,   6.17697923,
          3.13963703,  12.75389382,  28.89108219,  55.60654233,
          9.37784898,  17.5527011 ,  24.95924408,  60.15433186,
         25.2722066 ,  12.9145721 ,   5.62843737,   9.65020039],
       [ 27.27374209,   0.        ,  18.81213844,   2.59163637,
        217.29086371, 231.00403433,  17.05740923,  19.42552168,
         17.88260262,  10.20378457,   9.45792001,  31.15439452,
         16.04153695,  36.3340441 ,  22.3687295 ,  21.81958431,
         28.80163218,  15.9126602 ,  38.70778597,  42.89078594,
         33.20399778,  10.93991151,   3.65465706,  82.79836534,
         51.55235   ,  16.17310562,  21.69618507,  17.77672351],
       [ 10.50611665,  18.81213844,   0.        ,  16.34882661,
        206.04216594, 219.41427159,  2

### Preliminary: Visit duration by mean

In [107]:
# Dedicated POI avg visit in each POI
MIN_VISIT_MIN = 15

visits["taken"] = pd.to_datetime(visits["dateTaken"], unit= "s")
photo_count = visits["poiID"].value_counts().sort_index()

visits = visits.sort_values(["userID", "dateTaken"])
new_visits = (visits["poiID"] != visits["poiID"].shift()) | (visits["seqID"] != visits["seqID"].shift()) | (visits["userID"] != visits["userID"].shift())
visits["visitID"] = new_visits.cumsum()

visits_duration = visits.groupby("visitID").agg(poiID=("poiID", "first"), seqID=("seqID", "first"),
                                       userID=("userID", "first"), photos=("photoID", "size"),
                                       t_arrive=("dateTaken", "min"), t_depart=("dateTaken", "max"))

visits_duration["raw_min"] = (visits_duration["t_depart"] - visits_duration["t_arrive"]) / 60
visits_duration["minutes"] = visits_duration["raw_min"].clip(lower=MIN_VISIT_MIN)

per_poi = (visits_duration.groupby("poiID")
           .agg(visits=("minutes", "size"), users=("userID", "nunique"), 
                mean_min=("minutes", "mean"), median_min=("minutes", "median"))
           .join(poi.set_index("poiID")[["poiName", "theme"]])
           .join(photo_count.rename("photo")))

visits_by_mean = per_poi["mean_min"].loc[ids].to_numpy(float)

print("array length: ", len(visits_by_mean))

visits_by_mean

array length:  27


array([54.54788442, 28.34719101, 35.84000532, 23.49233333, 15.        ,
       15.        , 26.85915493, 24.33466969, 31.12832431, 21.52446483,
       21.05236111, 34.75043695, 59.37286325, 60.69866071, 23.13259109,
       17.41045752, 24.27529976, 24.50620748, 47.49125683, 40.28401826,
       33.09351852, 18.81270928, 26.18412073, 15.        , 39.01666667,
       39.24833333, 29.54122299])

### Helper function

In [ ]:
def total_walking_time(tour: tuple[int, ...]):
    depot = len(TRAVEL_TIME_MATRIX) - 1
    res = 0
    prev_node = depot
    
    for t in tour:
        res += TRAVEL_TIME_MATRIX[prev_node, t]
        prev_node = t
    
    res += TRAVEL_TIME_MATRIX[prev_node, depot]
    return res

def total_visit_time(tour: tuple[int, ...]):
    res = 0
    for t in tour:
        res += visits_by_mean[t]
    return res

def total_tour_time(tour):
    return total_walking_time(tour) + total_visit_time(tour)

### Genetic Algorithm method

In [ ]:
# Placeholder

problem = get_problem("g1")

algorithm = GA(
    pop_size=100,
    eliminate_duplicates=True)

res = minimize(problem,
               algorithm,
               seed=1,
               verbose=False)

print("Best solution found: \nX = %s\nF = %s" % (res.X, res.F))

Best solution found: 
X = [1.         1.         1.         1.         1.         1.
 1.         1.         1.         2.99221525 2.9943655  2.99054298
 1.        ]
F = [-14.97712372]
